# LIBRAS-UFOP — MediaPipe Holistic Landmark Extraction

This notebook extracts **LIBRAS-UFOP** landmarks from the RGB `Color.avi` videos and uses `labels.txt` to segment each video into individual samples.

Expected structure:

```text
ufop/
  labels.txt
  p1_c1_s1/
    Color.avi
    Depth.avi
    Skel/
  p1_c1_s2/
    Color.avi
    Depth.avi
    Skel/
  ...
```

Expected `labels.txt` format:

```text
p1_c1_s1 31-71:1 126-167:1 219-258:1 ...
```

Interpretation:

- `p1_c1_s1`: original directory/sequence;
- `31-71:1`: frames 31 through 71 belong to global class 1;
- each interval becomes a separate sample.

Main outputs:

```text
ufop_mediapipe_landmarks/
  metadata_ufop.csv
  ufop_mediapipe_all_videos_wide.csv
  ufop_segments_counts_by_sign.csv
  landmark_files/
    p01_c01_s01_seg000.parquet
    p01_c01_s01_seg001.parquet
    ...
```

## 1. Installation

Run this cell only when the required packages are not installed.

In [ ]:
%pip install -q mediapipe opencv-python pandas pyarrow tqdm numpy

## 2. Imports

In [ ]:
from pathlib import Path
import re
import time
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import mediapipe as mp
except ImportError as e:
    raise ImportError(
        "MediaPipe is not installed. Run: %pip install mediapipe opencv-python pandas pyarrow tqdm numpy"
    ) from e

## 3. Path configuration

Change only `UFOP_ROOT` when the dataset directory is elsewhere.

In [ ]:
# ==========================================================================
# EDIT HERE
# ==========================================================================

BASE_DIR = Path.cwd()
BASE_DIR = Path.cwd().parent
# LIBRAS-UFOP root directory.
# Expected structure:
# datasets/raw/ufop/labels.txt
# datasets/raw/ufop/p1_c1_s1/Color.avi
UFOP_ROOT = BASE_DIR / "data" / "raw" / "ufop"

# UFOP temporal segmentation file.
LABELS_PATH = UFOP_ROOT / "labels.txt"

# Output directory.
OUTPUT_ROOT = BASE_DIR / "data" / "interim" / "ufop_mediapipe_landmarks"
LANDMARKS_DIR = OUTPUT_ROOT / "landmark_files"

# Generated files.
METADATA_PATH = OUTPUT_ROOT / "metadata_ufop.csv"
FINAL_WIDE_CSV_PATH = OUTPUT_ROOT / "ufop_mediapipe_all_videos_wide.csv"
COUNTS_BY_SIGN_PATH = OUTPUT_ROOT / "ufop_segments_counts_by_sign.csv"
COUNTS_BY_PARTICIPANT_PATH = OUTPUT_ROOT / "ufop_segments_counts_by_participant.csv"
EXTRACTION_ERRORS_PATH = OUTPUT_ROOT / "ufop_extraction_errors.csv"

SAVE_FORMAT = "parquet"  # options: "parquet" or "csv"

MIN_DETECTION_CONFIDENCE = 0.4
MIN_TRACKING_CONFIDENCE = 0.4

SKIP_EXISTING = True

# Frame indices in labels.txt usually appear to be one-based.
# Set this to 0 only after confirming zero-based frame indices.
LABEL_FRAME_INDEX_BASE = 1

# False preserves NaN in the final wide CSV.
# True fills missing landmarks with 0.0, following some legacy dataset formats.
FILL_MISSING_WITH_0 = False

# Coordinates included in the final wide CSV.
# The raw long-format version also stores visibility.
COORDS_FOR_WIDE = ["x", "y", "z"]

LANDMARKS_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("UFOP_ROOT:", UFOP_ROOT)
print("Does UFOP_ROOT exist?", UFOP_ROOT.exists())
print("LABELS_PATH:", LABELS_PATH)
print("Does LABELS_PATH exist?", LABELS_PATH.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("LANDMARKS_DIR:", LANDMARKS_DIR)
print("METADATA_PATH:", METADATA_PATH)
print("FINAL_WIDE_CSV_PATH:", FINAL_WIDE_CSV_PATH)

if "/content" in str(OUTPUT_ROOT):
    raise ValueError("OUTPUT_ROOT still points to /content. When running locally on macOS, use BASE_DIR or Path.home().")

## 4. UFOP helper functions

In [ ]:
def clean_path_part(text):
    # Sanitize path components to generate safe filenames.
    text = str(text)
    text = text.replace(" ", "_").replace(".", "")
    text = re.sub(r"[^A-Za-z0-9_\-]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def normalize_sequence_id(sequence_id):
    # p1_c1_s1 -> p01_c01_s01
    match = re.match(r"^p(\d+)_c(\d+)_s(\d+)$", str(sequence_id).strip())
    if not match:
        return str(sequence_id).strip()
    p, c, s = match.groups()
    return f"p{int(p):02d}_c{int(c):02d}_s{int(s):02d}"


def parse_sequence_id(sequence_id):
    # Extract participant_id, category_id, and sign_id_in_category from p1_c1_s1.
    sequence_id = str(sequence_id).strip()
    match = re.match(r"^p(\d+)_c(\d+)_s(\d+)$", sequence_id)
    if not match:
        raise ValueError(f"Invalid sequence_id: {sequence_id}")

    participant_raw, category_raw, sign_raw = match.groups()

    return {
        "sequence_id_raw": sequence_id,
        "sequence_id": normalize_sequence_id(sequence_id),
        "participant_id": f"p{int(participant_raw):02d}",
        "category_id": f"c{int(category_raw):02d}",
        "sign_id_in_category": f"s{int(sign_raw):02d}",
        "participant_int": int(participant_raw),
        "category_int": int(category_raw),
        "sign_in_category_int": int(sign_raw),
    }


def get_ufop_sequence_dir(ufop_root, sequence_id_raw):
    ufop_root = Path(ufop_root)
    sequence_id_raw = str(sequence_id_raw)
    normalized = normalize_sequence_id(sequence_id_raw)

    candidates = [
        ufop_root / sequence_id_raw,
        ufop_root / normalized,
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return candidates[0]


def get_color_video_path(ufop_root, sequence_id_raw):
    sequence_dir = get_ufop_sequence_dir(ufop_root, sequence_id_raw)
    return sequence_dir / "Color.avi"


def safe_ufop_segment_output_name(sequence_id, segment_index, suffix=".parquet"):
    normalized = normalize_sequence_id(sequence_id)
    return f"{clean_path_part(normalized)}_seg{int(segment_index):03d}{suffix}"

## 5. Read `labels.txt` and build segment metadata

In [ ]:
def parse_ufop_labels(labels_path, ufop_root, save_format="parquet"):
    labels_path = Path(labels_path)
    ufop_root = Path(ufop_root)

    if not labels_path.exists():
        raise FileNotFoundError(f"labels.txt not found at: {labels_path}")

    suffix = ".parquet" if save_format == "parquet" else ".csv"
    rows = []

    with open(labels_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            parts = line.split()
            sequence_id_raw = parts[0]

            try:
                seq_info = parse_sequence_id(sequence_id_raw)
            except ValueError:
                print(f"Skipped line with invalid sequence_id: {line_number} -> {sequence_id_raw}")
                continue

            color_video_path = get_color_video_path(ufop_root, sequence_id_raw)

            for segment_index, token in enumerate(parts[1:]):
                segment_match = re.match(r"^(\d+)-(\d+):(\d+)$", token)

                if not segment_match:
                    print(f"Invalid segment on line {line_number}: {token}")
                    continue

                start_frame, end_frame, global_sign_id = segment_match.groups()
                start_frame = int(start_frame)
                end_frame = int(end_frame)
                global_sign_id = int(global_sign_id)

                if end_frame < start_frame:
                    print(f"Invalid interval on line {line_number}: {token}")
                    continue

                segment_output_name = safe_ufop_segment_output_name(
                    sequence_id=seq_info["sequence_id"],
                    segment_index=segment_index,
                    suffix=suffix,
                )

                landmark_path = Path("landmark_files") / segment_output_name
                sample_id = f"{seq_info['sequence_id']}_seg{segment_index:03d}"
                unique_video_name = f"{sample_id}_Color.avi"

                rows.append({
                    "sequence_id_raw": seq_info["sequence_id_raw"],
                    "sequence_id": seq_info["sequence_id"],
                    "participant_id": seq_info["participant_id"],
                    "participant_int": seq_info["participant_int"],
                    "category_id": seq_info["category_id"],
                    "category_int": seq_info["category_int"],
                    "sign_id_in_category": seq_info["sign_id_in_category"],
                    "sign_in_category_int": seq_info["sign_in_category_int"],
                    "sign_id": f"{global_sign_id:02d}",
                    "sign_id_int": global_sign_id,
                    "sign_key": f"{global_sign_id:02d}::{seq_info['category_id']}::{seq_info['sign_id_in_category']}",
                    "segment_index": int(segment_index),
                    "sample_id": sample_id,
                    "start_frame": start_frame,
                    "end_frame": end_frame,
                    "num_frames_label": end_frame - start_frame + 1,
                    "original_video_name": "Color.avi",
                    "video_name": unique_video_name,
                    "color_video_path": str(color_video_path),
                    "video_path": str(color_video_path),
                    "landmark_path": str(landmark_path),
                    "line_number": line_number,
                })

    metadata = pd.DataFrame(rows)

    if metadata.empty:
        raise ValueError("No segment was read from labels.txt.")

    metadata = metadata.sort_values([
        "participant_int",
        "category_int",
        "sign_id_int",
        "sequence_id",
        "segment_index",
    ]).reset_index(drop=True)

    return metadata


def build_ufop_metadata(labels_path, ufop_root, output_root, save_format="parquet"):
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    metadata = parse_ufop_labels(
        labels_path=labels_path,
        ufop_root=ufop_root,
        save_format=save_format,
    )

    metadata["color_video_exists"] = metadata["color_video_path"].apply(lambda p: Path(p).exists())
    metadata.to_csv(METADATA_PATH, index=False)

    print(f"Metadata saved to: {METADATA_PATH}")
    print(f"Total segments/samples: {len(metadata)}")
    print(f"Total sequences/directories: {metadata['sequence_id'].nunique()}")
    print(f"Total participants: {metadata['participant_id'].nunique()}")
    print(f"Total categories: {metadata['category_id'].nunique()}")
    print(f"Total global classes: {metadata['sign_id'].nunique()}")
    print(f"Missing Color.avi videos: {(~metadata['color_video_exists']).sum()}")

    return metadata

### 5.1 Generate metadata

In [ ]:
metadata_ufop = build_ufop_metadata(
    labels_path=LABELS_PATH,
    ufop_root=UFOP_ROOT,
    output_root=OUTPUT_ROOT,
    save_format=SAVE_FORMAT,
)

metadata_ufop.head()

### 5.2 Dataset diagnostics

In [ ]:
summary = {
    "segments": len(metadata_ufop),
    "sequences": metadata_ufop["sequence_id"].nunique(),
    "participants": metadata_ufop["participant_id"].nunique(),
    "categories": metadata_ufop["category_id"].nunique(),
    "classes": metadata_ufop["sign_id"].nunique(),
    "min_sign_id": metadata_ufop["sign_id_int"].min(),
    "max_sign_id": metadata_ufop["sign_id_int"].max(),
}
summary

In [ ]:
classes_ufop = (
    metadata_ufop[["sign_id", "sign_key", "category_id", "sign_id_in_category"]]
    .drop_duplicates()
    .sort_values("sign_id")
)

print("Total classes:", len(classes_ufop))
classes_ufop.head(20)

In [ ]:
all_ids = set(range(metadata_ufop["sign_id_int"].min(), metadata_ufop["sign_id_int"].max() + 1))
found_ids = set(metadata_ufop["sign_id_int"].unique())
missing_global_ids = sorted(all_ids - found_ids)
print("Missing global IDs in the min..max range:", missing_global_ids)

In [ ]:
counts_by_sign = (
    metadata_ufop
    .groupby(["sign_id", "sign_key", "category_id", "sign_id_in_category"])
    .size()
    .reset_index(name="total_segments")
    .sort_values("sign_id")
)
counts_by_sign.to_csv(COUNTS_BY_SIGN_PATH, index=False)
counts_by_sign.head()

In [ ]:
counts_by_participant = (
    metadata_ufop
    .groupby("participant_id")
    .size()
    .reset_index(name="total_segments")
    .sort_values("participant_id")
)
counts_by_participant.to_csv(COUNTS_BY_PARTICIPANT_PATH, index=False)
counts_by_participant

In [ ]:
missing_videos = metadata_ufop[~metadata_ufop["color_video_exists"]]
print("Segments with missing Color.avi:", len(missing_videos))
missing_videos[["sequence_id", "color_video_path"]].drop_duplicates().head(20)

## 6. MediaPipe Holistic functions

In [ ]:
mp_holistic = mp.solutions.holistic

LANDMARK_GROUPS = {
    "hand_0": 21,  # left hand
    "hand_1": 21,  # right hand
    "pose": 33,
    "face": 468,
}


def get_landmark_list(results, landmark_type):
    if landmark_type == "face":
        return results.face_landmarks
    if landmark_type == "pose":
        return results.pose_landmarks
    if landmark_type == "hand_0":
        return results.left_hand_landmarks
    if landmark_type == "hand_1":
        return results.right_hand_landmarks
    raise ValueError(f"Invalid landmark type: {landmark_type}")


def landmarks_to_rows(results, frame_id, original_frame_id, timestamp_ms, video_name, sample_id):
    rows = []

    for landmark_type, expected_count in LANDMARK_GROUPS.items():
        landmark_list = get_landmark_list(results, landmark_type)

        if landmark_list is None:
            for landmark_index in range(expected_count):
                rows.append({
                    "sample_id": sample_id,
                    "video_name": video_name,
                    "frame_id": frame_id,
                    "original_frame_id": original_frame_id,
                    "timestamp_ms": timestamp_ms,
                    "type": landmark_type,
                    "landmark_index": landmark_index,
                    "x": np.nan,
                    "y": np.nan,
                    "z": np.nan,
                    "visibility": np.nan,
                })
            continue

        landmarks = landmark_list.landmark

        for landmark_index in range(expected_count):
            if landmark_index < len(landmarks):
                lm = landmarks[landmark_index]
                rows.append({
                    "sample_id": sample_id,
                    "video_name": video_name,
                    "frame_id": frame_id,
                    "original_frame_id": original_frame_id,
                    "timestamp_ms": timestamp_ms,
                    "type": landmark_type,
                    "landmark_index": landmark_index,
                    "x": lm.x,
                    "y": lm.y,
                    "z": lm.z,
                    "visibility": getattr(lm, "visibility", np.nan),
                })
            else:
                rows.append({
                    "sample_id": sample_id,
                    "video_name": video_name,
                    "frame_id": frame_id,
                    "original_frame_id": original_frame_id,
                    "timestamp_ms": timestamp_ms,
                    "type": landmark_type,
                    "landmark_index": landmark_index,
                    "x": np.nan,
                    "y": np.nan,
                    "z": np.nan,
                    "visibility": np.nan,
                })

    return rows

## 7. Landmark extraction by segment

In [ ]:
def save_landmark_dataframe(df, output_path, save_format="parquet"):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if save_format == "parquet":
        df.to_parquet(output_path, index=False)
    elif save_format == "csv":
        df.to_csv(output_path, index=False)
    else:
        raise ValueError("save_format must be either 'parquet' or 'csv'")


def extract_mediapipe_from_video_segment(
    video_path,
    output_path,
    sample_id,
    video_name,
    start_frame,
    end_frame,
    label_frame_index_base=1,
    min_detection_confidence=0.4,
    min_tracking_confidence=0.4,
    save_format="parquet",
):
    video_path = Path(video_path)
    output_path = Path(output_path)

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:
        fps = 30

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # OpenCV uses zero-based indices. The labels.txt file usually appears to use one-based indices.
    opencv_start_frame = max(0, int(start_frame) - int(label_frame_index_base))
    opencv_end_frame = max(0, int(end_frame) - int(label_frame_index_base))

    if opencv_start_frame >= total_frames:
        cap.release()
        raise ValueError(f"start_frame is outside the video: start={start_frame}, total_frames={total_frames}, video={video_path}")

    opencv_end_frame = min(opencv_end_frame, total_frames - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, opencv_start_frame)

    all_rows = []
    frame_id = 0
    current_opencv_frame = opencv_start_frame

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        enable_segmentation=False,
        smooth_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=min_detection_confidence,
        min_tracking_confidence=min_tracking_confidence,
    ) as holistic:
        while current_opencv_frame <= opencv_end_frame:
            success, frame_bgr = cap.read()
            if not success:
                break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frame_rgb.flags.writeable = False
            results = holistic.process(frame_rgb)

            original_frame_id = current_opencv_frame + int(label_frame_index_base)
            timestamp_ms = int((current_opencv_frame / fps) * 1000)

            rows = landmarks_to_rows(
                results=results,
                frame_id=frame_id,
                original_frame_id=original_frame_id,
                timestamp_ms=timestamp_ms,
                video_name=video_name,
                sample_id=sample_id,
            )
            all_rows.extend(rows)
            frame_id += 1
            current_opencv_frame += 1

    cap.release()

    df = pd.DataFrame(all_rows)
    save_landmark_dataframe(df, output_path, save_format=save_format)

    return {
        "frames_extracted": frame_id,
        "total_rows": len(df),
        "total_frames_video": total_frames,
        "fps": fps,
    }

## 8. Read landmark files

In [ ]:
def read_landmark_file(file_path):
    file_path = Path(file_path)
    if file_path.suffix.lower() == ".parquet":
        return pd.read_parquet(file_path)
    if file_path.suffix.lower() == ".csv":
        return pd.read_csv(file_path)
    raise ValueError(f"Unsupported format: {file_path}")

## 9. Full UFOP extraction

In [ ]:
def extract_ufop_dataset(
    metadata,
    output_root,
    save_format="parquet",
    min_detection_confidence=0.4,
    min_tracking_confidence=0.4,
    label_frame_index_base=1,
    skip_existing=True,
):
    output_root = Path(output_root)
    metadata = metadata.copy()

    statuses = []
    frames_extracted_list = []
    total_rows_list = []
    error_rows = []

    for idx, row in tqdm(metadata.iterrows(), total=len(metadata)):
        video_path = Path(row["color_video_path"])
        output_path = output_root / row["landmark_path"]

        if skip_existing and output_path.exists():
            statuses.append("skipped_existing")
            try:
                df_existing = read_landmark_file(output_path)
                frames_extracted_list.append(df_existing["frame_id"].nunique())
                total_rows_list.append(len(df_existing))
            except Exception:
                frames_extracted_list.append(np.nan)
                total_rows_list.append(np.nan)
            continue

        try:
            result = extract_mediapipe_from_video_segment(
                video_path=video_path,
                output_path=output_path,
                sample_id=row["sample_id"],
                video_name=row["video_name"],
                start_frame=row["start_frame"],
                end_frame=row["end_frame"],
                label_frame_index_base=label_frame_index_base,
                min_detection_confidence=min_detection_confidence,
                min_tracking_confidence=min_tracking_confidence,
                save_format=save_format,
            )
            statuses.append("processed")
            frames_extracted_list.append(result["frames_extracted"])
            total_rows_list.append(result["total_rows"])

        except Exception as e:
            statuses.append("error")
            frames_extracted_list.append(np.nan)
            total_rows_list.append(np.nan)
            error_rows.append({
                "sample_id": row.get("sample_id", ""),
                "sequence_id": row.get("sequence_id", ""),
                "video_path": str(video_path),
                "start_frame": row.get("start_frame", ""),
                "end_frame": row.get("end_frame", ""),
                "error": repr(e),
            })
            print(f"Error while processing {row.get('sample_id', '')}: {e}")

    metadata["status"] = statuses
    metadata["frames_extracted"] = frames_extracted_list
    metadata["total_landmark_rows"] = total_rows_list
    metadata.to_csv(METADATA_PATH, index=False)

    if error_rows:
        pd.DataFrame(error_rows).to_csv(EXTRACTION_ERRORS_PATH, index=False)
        print(f"Errors saved to: {EXTRACTION_ERRORS_PATH}")

    print(f"Updated metadata saved to: {METADATA_PATH}")
    print(metadata["status"].value_counts(dropna=False))
    return metadata

### 9.1 Estimate runtime with a sample

In [ ]:
def estimate_processing_time_ufop(metadata, sample_size=10):
    sample = metadata.head(sample_size)
    times = []

    print(f"Estimating runtime with {len(sample)} segments...")

    for _, row in sample.iterrows():
        output_path = OUTPUT_ROOT / row["landmark_path"]
        start = time.time()
        extract_mediapipe_from_video_segment(
            video_path=row["color_video_path"],
            output_path=output_path,
            sample_id=row["sample_id"],
            video_name=row["video_name"],
            start_frame=row["start_frame"],
            end_frame=row["end_frame"],
            label_frame_index_base=LABEL_FRAME_INDEX_BASE,
            min_detection_confidence=MIN_DETECTION_CONFIDENCE,
            min_tracking_confidence=MIN_TRACKING_CONFIDENCE,
            save_format=SAVE_FORMAT,
        )
        elapsed = time.time() - start
        times.append(elapsed)
        print(f"{row['sample_id']}: {elapsed:.2f}s")

    avg_time = float(np.mean(times))
    estimated_total_seconds = avg_time * len(metadata)

    print("\nResumo")
    print(f"Average time per segment: {avg_time:.2f}s")
    print(f"Total segments: {len(metadata)}")
    print(f"Estimated total time: {estimated_total_seconds / 60:.2f} minutes")
    print(f"Estimated total time: {estimated_total_seconds / 3600:.2f} hours")

# Uncomment to test:
# estimate_processing_time_ufop(metadata_ufop, sample_size=10)

### 9.2 Run the full extraction

In [ ]:
metadata_ufop = extract_ufop_dataset(
    metadata=metadata_ufop,
    output_root=OUTPUT_ROOT,
    save_format=SAVE_FORMAT,
    min_detection_confidence=MIN_DETECTION_CONFIDENCE,
    min_tracking_confidence=MIN_TRACKING_CONFIDENCE,
    label_frame_index_base=LABEL_FRAME_INDEX_BASE,
    skip_existing=SKIP_EXISTING,
)

metadata_ufop.head()

## 10. Convert long format to wide format

In [ ]:
def compute_missing_flags(df_long):
    missing_rows = []

    for frame_id, frame_df in df_long.groupby("frame_id"):
        hand_df = frame_df[frame_df["type"].isin(["hand_0", "hand_1"])]
        face_df = frame_df[frame_df["type"].eq("face")]
        pose_df = frame_df[frame_df["type"].eq("pose")]

        missing_hand = hand_df[["x", "y"]].isna().any().any()
        missing_face = face_df[["x", "y"]].isna().any().any()
        missing_pose = pose_df[["x", "y"]].isna().any().any()
        original_frame_id = frame_df["original_frame_id"].iloc[0] if "original_frame_id" in frame_df.columns else np.nan

        missing_rows.append({
            "frame_id": frame_id,
            "original_frame_id": original_frame_id,
            "missing_hand": bool(missing_hand),
            "missing_face": bool(missing_face),
            "missing_pose": bool(missing_pose),
        })

    return pd.DataFrame(missing_rows)


def expected_wide_columns(coords=("x", "y", "z")):
    cols = []
    for landmark_type, count in LANDMARK_GROUPS.items():
        for landmark_index in range(count):
            for coord in coords:
                cols.append(f"{landmark_type}_{landmark_index}_{coord}")
    return cols


def long_to_wide_one_segment(
    df_long,
    metadata_row,
    fill_missing_with_0=False,
    coords_for_wide=("x", "y", "z"),
):
    df = df_long.copy()
    required = {"frame_id", "type", "landmark_index"}
    missing_required = required - set(df.columns)
    if missing_required:
        raise ValueError(f"df_long is missing required columns: {missing_required}")

    if "original_frame_id" not in df.columns:
        df["original_frame_id"] = np.nan

    df["landmark_name"] = df["type"].astype(str) + "_" + df["landmark_index"].astype(str)
    missing_df = compute_missing_flags(df)

    wide_parts = []
    for coord in coords_for_wide:
        pivot = df.pivot_table(
            index=["frame_id", "original_frame_id"],
            columns="landmark_name",
            values=coord,
            aggfunc="first",
        )
        pivot.columns = [f"{col}_{coord}" for col in pivot.columns]
        wide_parts.append(pivot)

    df_wide = pd.concat(wide_parts, axis=1).reset_index()

    for col in expected_wide_columns(coords_for_wide):
        if col not in df_wide.columns:
            df_wide[col] = np.nan

    ordered_cols = ["frame_id", "original_frame_id"] + expected_wide_columns(coords_for_wide)
    df_wide = df_wide[ordered_cols]

    if fill_missing_with_0:
        landmark_cols = expected_wide_columns(coords_for_wide)
        df_wide[landmark_cols] = df_wide[landmark_cols].fillna(0.0)

    meta_cols = {
        "sign_id": metadata_row["sign_id"],
        "sign_key": metadata_row["sign_key"],
        "category_id": metadata_row["category_id"],
        "sign_id_in_category": metadata_row["sign_id_in_category"],
        "participant_id": metadata_row["participant_id"],
        "sample_id": metadata_row["sample_id"],
        "sequence_id": metadata_row["sequence_id"],
        "video_name": metadata_row["video_name"],
        "start_frame": metadata_row["start_frame"],
        "end_frame": metadata_row["end_frame"],
    }

    for col_name, value in reversed(list(meta_cols.items())):
        df_wide.insert(0, col_name, value)

    df_wide = df_wide.merge(missing_df, on=["frame_id", "original_frame_id"], how="left")

    for col in ["missing_hand", "missing_face", "missing_pose"]:
        df_wide[col] = df_wide[col].fillna(False).astype(bool)

    return df_wide

## 11. Generate one MINDS-style wide CSV

In [ ]:
def build_single_wide_csv_from_ufop_metadata(
    metadata_path,
    output_root,
    final_csv_path,
    fill_missing_with_0=False,
    overwrite=True,
):
    metadata_path = Path(metadata_path)
    output_root = Path(output_root)
    final_csv_path = Path(final_csv_path)
    metadata = pd.read_csv(metadata_path)

    if overwrite and final_csv_path.exists():
        final_csv_path.unlink()

    first_write = True
    written_segments = 0
    skipped_segments = 0

    for _, row in tqdm(metadata.iterrows(), total=len(metadata)):
        if "status" in metadata.columns and row["status"] == "error":
            skipped_segments += 1
            continue

        landmark_path = output_root / row["landmark_path"]

        if not landmark_path.exists():
            print(f"File not found: {landmark_path}")
            skipped_segments += 1
            continue

        df_long = read_landmark_file(landmark_path)
        df_wide = long_to_wide_one_segment(
            df_long=df_long,
            metadata_row=row,
            fill_missing_with_0=fill_missing_with_0,
            coords_for_wide=COORDS_FOR_WIDE,
        )

        df_wide.to_csv(
            final_csv_path,
            mode="w" if first_write else "a",
            header=first_write,
            index=False,
        )

        first_write = False
        written_segments += 1

    print(f"Final CSV saved to: {final_csv_path}")
    print(f"Written segments: {written_segments}")
    print(f"Skipped segments: {skipped_segments}")
    return final_csv_path

In [ ]:
build_single_wide_csv_from_ufop_metadata(
    metadata_path=METADATA_PATH,
    output_root=OUTPUT_ROOT,
    final_csv_path=FINAL_WIDE_CSV_PATH,
    fill_missing_with_0=FILL_MISSING_WITH_0,
    overwrite=True,
)

## 12. Inspect the final CSV

In [ ]:
print("Does the final CSV exist?", FINAL_WIDE_CSV_PATH.exists())
print("Path:", FINAL_WIDE_CSV_PATH)

df_preview = pd.read_csv(FINAL_WIDE_CSV_PATH, nrows=5)
df_preview

In [ ]:
df_preview[[
    "sign_id",
    "sign_key",
    "category_id",
    "sign_id_in_category",
    "participant_id",
    "sample_id",
    "sequence_id",
    "video_name",
    "frame_id",
    "original_frame_id",
    "missing_hand",
    "missing_face",
    "missing_pose",
]]

## 13. Final reports

In [ ]:
metadata_final = pd.read_csv(METADATA_PATH)

print("Extraction status:")
print(metadata_final["status"].value_counts(dropna=False) if "status" in metadata_final.columns else "No status column")

print("Segments per participant:")
print(metadata_final.groupby("participant_id").size())

print("Segments per category:")
print(metadata_final.groupby("category_id").size())

print("Global classes:", metadata_final["sign_id"].nunique())

In [ ]:
counts_by_sign = (
    metadata_final
    .groupby(["sign_id", "sign_key", "category_id", "sign_id_in_category"])
    .size()
    .reset_index(name="total_segments")
    .sort_values("sign_id")
)

counts_by_sign.to_csv(COUNTS_BY_SIGN_PATH, index=False)
counts_by_sign.head(20)

## 14. Optional: save a Parquet version of the wide CSV

In [ ]:
# Useful when the CSV becomes very large.
# FINAL_WIDE_PARQUET_PATH = OUTPUT_ROOT / "ufop_mediapipe_all_videos_wide.parquet"
# df_all = pd.read_csv(FINAL_WIDE_CSV_PATH)
# df_all.to_parquet(FINAL_WIDE_PARQUET_PATH, index=False)
# FINAL_WIDE_PARQUET_PATH

## 15. Important notes

- The notebook uses only `Color.avi` because its purpose is to extract MediaPipe landmarks from RGB video.
- `Depth.avi` and `Skel/` are original LIBRAS-UFOP modalities, but they are not used in this extraction.
- Each interval in `labels.txt` becomes an individual sample.
- `frame_id` is relative to the segment and starts at 0.
- `original_frame_id` preserves the original video frame index defined by `labels.txt`.
- `FILL_MISSING_WITH_0 = False` preserves `NaN`, which is recommended for later interpolation.
- To generate a file compatible with legacy code that expects `0.0`, set `FILL_MISSING_WITH_0 = True` and regenerate the wide CSV.